# Mercury 3: watch a diffusion LLM revise its own answer

Two live experiments from the tutorial. Both run against a synthetic
customer-support case: a customer says they were charged twice for one order.

**What this costs you**

| | Calls | Approx. tokens | Time |
|---|---:|---:|---:|
| 1. Diffusion replay | 1 | ~2,000 | ~3 s |
| 2. The token-budget trap | 2 | ~2,500 | ~7 s |
| **Total** | **3** | **~4,500** | **~10 s** |

You need an Inception API key from [platform.inceptionlabs.ai](https://platform.inceptionlabs.ai).

**Your numbers will not match the article's.** That is the point of the
tutorial's section 6: a single run of anything tells you very little. The
article reports medians over five trials; this notebook runs once so it stays
cheap.

## Setup

In [ ]:
!pip install -q inceptionai

import getpass, json, os, time, difflib
from inceptionai import Inception
from IPython.display import clear_output

if not os.environ.get("INCEPTION_API_KEY"):
    os.environ["INCEPTION_API_KEY"] = getpass.getpass("Inception API key: ")

client = Inception()
MODEL = "mercury-3"
print("ready")

## The case file

Two payment rows 94 seconds apart with **different** authorization codes. A
fulfillment log showing a second reservation that was never attached to an order.
A customer history where the same complaint ended with both charges settling.

The one fact that would settle it — when the pending authorization expires — is
deliberately missing. **No model can answer this case with certainty.** Watch
whether yours admits that.

In [ ]:
CASE = {
    "scenario": "Customer reports two charges for one order",
    "customer_message": "I ordered one pair of headphones yesterday evening and my card now shows two charges of $129 each. I have one confirmation email and one tracking number. I only pressed buy once. Please refund the second charge today, I cannot afford to be out $258.",
    "order": {
        "order_id": "ORD-1048",
        "placed_at": "2026-10-28T19:41:08Z",
        "status": "shipped",
        "shipped_at": "2026-10-29T08:12:00Z",
        "item": "wireless headphones",
        "quantity": 1,
        "amount_usd": 129,
        "orders_on_account_that_day": 1
    },
    "payment_records": [
        {
            "entry": 1,
            "at": "2026-10-28T19:41:11Z",
            "amount_usd": 129,
            "status": "completed",
            "authorization_code": "A7X41K",
            "card_last_four": "4417"
        },
        {
            "entry": 2,
            "at": "2026-10-28T19:42:45Z",
            "amount_usd": 129,
            "status": "pending",
            "authorization_code": "B2M09T",
            "card_last_four": "4417"
        }
    ],
    "fulfillment_log": [
        "19:41:12 reserved 1 unit for ORD-1048",
        "19:42:47 reserved 1 unit (no order id attached)",
        "19:58:30 released 1 unreserved unit back to stock",
        "08:12:00 shipped 1 unit against ORD-1048, one tracking number issued"
    ],
    "customer_history": [
        {
            "date": "2026-10-07",
            "contact_reason": "reported two charges for one order",
            "outcome": "both charges settled; agent issued a refund for one of them"
        },
        {
            "date": "2026-06-19",
            "contact_reason": "delivery address change",
            "outcome": "resolved, no payment involved"
        }
    ],
    "support_policy": [
        "A pending card authorization can appear on a customer statement as a second charge.",
        "A pending authorization that is never captured usually disappears within 3-5 business days.",
        "Escalate to the payments team if both charges settle.",
        "Agents may not issue a refund against an authorization that has not settled.",
        "Tell the customer what you have confirmed and what you have not."
    ],
    "available_agent_actions": [
        "explain the pending authorization to the customer",
        "set a follow-up for a named number of business days",
        "escalate to the payments team",
        "issue a refund for a settled charge",
        "request the authorization expiry date from the payment processor"
    ]
}

SYSTEM = """You are a customer-support escalation assistant.
Use only the evidence in the supplied case file.
Separate what the evidence shows from what you infer.
Do not state that a refund has been issued unless the case file says so.
If the evidence is insufficient to decide, say so explicitly."""

TASK = """Analyze this customer-support case.

Return:
1. The most likely explanation.
2. Three pieces of evidence from the case file that support it.
3. Two alternative explanations, and what is still uncertain.
4. The next three actions for the agent, in priority order.

Keep the answer concise. Write it so the customer could read it."""

MESSAGES = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": f"{TASK}\n\nCASE FILE:\n{json.dumps(CASE, indent=2)}"},
]

print(f"prompt: {len(MESSAGES[0]['content']) + len(MESSAGES[1]['content'])} chars")

---

## Experiment 1 — watch the answer denoise

`diffusing=True` makes the API send a **state** on every chunk: the whole current
answer, not the next piece of it. So you overwrite the display instead of
appending to it.

The real run finishes in under a second, so we collect every state first and
replay it slowly afterwards.

In [ ]:
states = []
t0 = time.perf_counter()

for chunk in client.chat.completions.create(
    model=MODEL,
    messages=MESSAGES,
    reasoning_effort="low",
    max_completion_tokens=8192,
    temperature=0.75,
    stream=True,
    diffusing=True,
):
    text = chunk.choices[0].delta.content or ""
    if not text:
        continue                      # exactly 2 chunks carry no text
    states.append({"t": time.perf_counter() - t0, "text": text})

print(f"{len(states)} refinement states in {states[-1]['t']:.3f} s")
print(f"first state: {len(states[0]['text'])} chars"
      f"  ->  final: {len(states[-1]['text'])} chars")

Now replay it. Changed text is highlighted.

In [ ]:
DIM, HI, OFF = "\033[2m", "\033[46;30m", "\033[0m"

def lcp(a, b):
    """Length of the longest common prefix of two strings."""
    n = 0
    for x, y in zip(a, b):
        if x != y:
            break
        n += 1
    return n

for i, s in enumerate(states):
    clear_output(wait=True)
    keep = lcp(states[i - 1]["text"], s["text"]) if i else 0
    print(f"state {i + 1}/{len(states)}   {len(s['text']):,} chars   "
          f"t = {s['t']:.3f} s   first edit at char {keep + 1}\n")
    print(DIM + s["text"][:keep] + OFF + HI + s["text"][keep:] + OFF)
    time.sleep(1.4)

### The proof: it is not appending

If Mercury 3 were writing left to right, every state would **start with** the
complete state before it. Check that directly.

A pure append means the common prefix covers all of the previous state. Anything
less means the model went back and changed text it had already written.

In [ ]:
final = states[-1]["text"]
appends = 0

print(f"{'state':>6} {'chars':>8} {'1st edit':>9} {'rewrote':>9} {'vs final':>9}  kind")
print("-" * 60)
for i in range(1, len(states)):
    a, b = states[i - 1]["text"], states[i]["text"]
    n = lcp(a, b)
    is_append = (n == len(a) and len(b) > len(a))
    appends += is_append
    sim = difflib.SequenceMatcher(None, b, final).ratio() * 100
    print(f"{i:>6} {len(b):>8,} {n + 1:>9,} {len(a) - n:>9,} {sim:>8.1f}%  "
          f"{'APPEND' if is_append else 'rewrite'}")

print("-" * 60)
print(f"pure appends: {appends} of {len(states) - 1} transitions")
print(f"the final state rewrote {len(final) - lcp(states[-2]['text'], final):,} "
      f"of its {len(final):,} characters")

Our run got **0 of 11**. All eleven diverged within the first **14 characters**
of a 2,636-character answer, six of them after a single character — the model
was still rewriting its opening words when the answer was already 2,249
characters long.

An autoregressive model cannot do this. A committed token stays committed.



In [ ]:
print(states[-1]["text"])

---

## Experiment 2 — the token-budget trap

`reasoning_effort` makes Mercury think before it answers, and **those thinking
tokens count against `max_completion_tokens`**. Inception's docs do not say so.

Set the cap too low and at `high` effort the budget is gone before a single word
of the answer is emitted. You get `content: None` and **no error**.

Two calls at a deliberately low cap of 1,600.

In [ ]:
def call(effort, cap):
    r = client.chat.completions.create(
        model=MODEL,
        messages=MESSAGES,
        reasoning_effort=effort,
        max_completion_tokens=cap,
        temperature=0.75,
    )
    u = r.usage
    rt = u.completion_tokens_details.reasoning_tokens
    text = r.choices[0].message.content          # may be None
    return {
        "effort": effort,
        "finish_reason": r.choices[0].finish_reason,
        "completion_tokens": u.completion_tokens,
        "reasoning_tokens": rt,
        "visible_tokens": u.completion_tokens - rt,
        "answer": text,
    }

for effort in ("low", "high"):
    r = call(effort, cap=1600)
    print(f"=== {effort.upper()} at max_completion_tokens=1600 ===")
    print(f"  finish_reason     : {r['finish_reason']}")
    print(f"  completion tokens : {r['completion_tokens']:,}")
    print(f"  of which reasoning: {r['reasoning_tokens']:,}"
          f"  ({r['reasoning_tokens'] / max(r['completion_tokens'], 1):.0%})")
    head = (r["answer"] or "").strip().splitlines()
    print(f"  answer            : {head[0][:60] if head else '*** None ***'}")
    print()

On our run, `high` spent **1,539 of 1,539 tokens on reasoning** and returned
`content: None` with `finish_reason: "length"`. `low` had budget to spare and
answered normally.

Note what did *not* happen: no exception, no warning. If you had been timing this
call you would have recorded a fast, cheap, completely empty response.

**The rule:** check `finish_reason` and
`completion_tokens_details.reasoning_tokens` on every call. Inception's own
quick-start uses 8192 — treat that as the floor.

Run it again with a sane cap to see the difference:

In [ ]:
r = call("high", cap=8192)
print(f"finish_reason     : {r['finish_reason']}")
print(f"completion tokens : {r['completion_tokens']:,}")
print(f"  reasoning       : {r['reasoning_tokens']:,}  (hidden from you)")
print(f"  visible answer  : {r['visible_tokens']:,}")
print()
print(r["answer"][:400], "...")

---

## What to try next

- Change `reasoning_effort` to `instant` and `medium` and compare the
  reasoning-token counts. The visible answer length moves far less than you
  would expect — in the article's five-trial test the difference between `low`
  and `high` was not detectable at all (p = 0.15).
- Delete the `customer_history` block from `CASE` and re-run experiment 1. The
  model loses its reason to hedge about the charge settling.
- Add the authorization expiry date to the case file. The uncertainty the model
  keeps flagging should disappear.
- Run the diffusion replay five times and count the states. We got 7 to 12.

The full harness, with the five-trial modes used for the article's numbers, is
`mercury3_incident_demo.py` in the repo:

```bash
python mercury3_incident_demo.py --mode compare --repeat 5 --incident case_support.json
```

That one costs 10 calls and about 31,000 tokens, which is why it is not in this
notebook.